
# Roxy notebook example: Dipeptide composition (DPC)

This notebook is a **reference implementation example** for the **DPC descriptor family** in Roxy.

Dipeptide composition is one of the most important classical sequence descriptor families because it captures **local sequential context** beyond single-residue composition. Instead of only asking *which amino acids are present*, DPC also asks *which adjacent amino-acid pairs appear* and how often.

## Covered outputs

This notebook implements:

- sequence cleaning
- extraction of observed dipeptides
- absolute dipeptide counts
- normalized dipeptide frequencies
- ordered 400-dimensional DPC vectors
- handling of short sequences
- dataset-level summaries
- class-style implementation for later migration into Roxy

The notebook is designed as a **clean implementation example** so the student can directly adapt it into the real package.


In [1]:

from collections import Counter
from itertools import product

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "dpc_1",
            "dpc_2",
            "dpc_3",
            "dpc_4",
            "dpc_5",
            "dpc_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,dpc_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,dpc_2,GGGGGGGGGGGGGGG,B
2,dpc_3,KRRKRRKRRKRRDDDDEE,A
3,dpc_4,ACDEFGHIKLMNPQRSTVWY,B
4,dpc_5,PPPPGSSSSSTTTTNNQQQ,A
5,dpc_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)

ALL_DIPEPTIDES = ["".join(dp) for dp in product(STANDARD_AA, repeat=2)]
len(ALL_DIPEPTIDES), ALL_DIPEPTIDES[:10]


(400, ['AA', 'AC', 'AD', 'AE', 'AF', 'AG', 'AH', 'AI', 'AK', 'AL'])

## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    """Clean a protein sequence, keeping only the 20 standard amino acids."""
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    seq = "".join([aa for aa in seq if aa in STANDARD_AA_SET])
    return seq


def extract_dipeptides(seq: str) -> list[str]:
    """Return all adjacent dipeptides observed in the cleaned sequence."""
    seq = clean_sequence(seq)
    if len(seq) < 2:
        return []
    return [seq[i:i+2] for i in range(len(seq) - 1)]


def dipeptide_count_dict(seq: str) -> dict:
    """Return absolute counts for all 400 possible dipeptides."""
    dipeptides = extract_dipeptides(seq)
    counts = Counter(dipeptides)
    return {f"dpc_count_{dp}": counts.get(dp, 0) for dp in ALL_DIPEPTIDES}


def dipeptide_frequency_dict(seq: str) -> dict:
    """Return normalized frequencies for all 400 possible dipeptides."""
    dipeptides = extract_dipeptides(seq)
    total = len(dipeptides)
    if total == 0:
        return {f"dpc_freq_{dp}": np.nan for dp in ALL_DIPEPTIDES}
    counts = Counter(dipeptides)
    return {f"dpc_freq_{dp}": counts.get(dp, 0) / total for dp in ALL_DIPEPTIDES}


def dipeptide_composition(
    seq: str,
    include_counts: bool = True,
    include_frequencies: bool = True,
) -> dict:
    """Compute a combined DPC descriptor dictionary."""
    seq = clean_sequence(seq)
    dipeptides = extract_dipeptides(seq)

    out = {
        "dpc_length": len(seq),
        "dpc_valid_residue_count": len(seq),
        "dpc_total_dipeptides": len(dipeptides),
        "dpc_unique_dipeptides": len(set(dipeptides)),
    }

    if include_counts:
        out.update(dipeptide_count_dict(seq))
    if include_frequencies:
        out.update(dipeptide_frequency_dict(seq))

    if include_frequencies and len(dipeptides) > 0:
        out["dpc_frequency_sum"] = sum(out[f"dpc_freq_{dp}"] for dp in ALL_DIPEPTIDES)
    else:
        out["dpc_frequency_sum"] = np.nan

    return out


## Functional usage on one sequence

In [5]:

example = dipeptide_composition(df_demo.loc[0, "sequence"])
list(example.items())[:12]


[('dpc_length', 24),
 ('dpc_valid_residue_count', 24),
 ('dpc_total_dipeptides', 23),
 ('dpc_unique_dipeptides', 22),
 ('dpc_count_AA', 0),
 ('dpc_count_AC', 0),
 ('dpc_count_AD', 0),
 ('dpc_count_AE', 0),
 ('dpc_count_AF', 0),
 ('dpc_count_AG', 0),
 ('dpc_count_AH', 0),
 ('dpc_count_AI', 0)]

## Observed dipeptides in one example

In [6]:

example_seq = df_demo.loc[0, "sequence"]
example_dipeptides = extract_dipeptides(example_seq)
example_dipeptides[:15], len(example_dipeptides)


(['MK',
  'KW',
  'WV',
  'VT',
  'TF',
  'FI',
  'IS',
  'SL',
  'LL',
  'LF',
  'FL',
  'LF',
  'FS',
  'SS',
  'SA'],
 23)

## Apply DPC to the full demo dataset

In [7]:

df_dpc = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(dipeptide_composition).apply(pd.Series),
    ],
    axis=1,
)

df_dpc.head()


,sequence_id,sequence,label,dpc_length,dpc_valid_residue_count,dpc_total_dipeptides,dpc_unique_dipeptides,dpc_count_AA,dpc_count_AC,dpc_count_AD,...,dpc_freq_YN,dpc_freq_YP,dpc_freq_YQ,dpc_freq_YR,dpc_freq_YS,dpc_freq_YT,dpc_freq_YV,dpc_freq_YW,dpc_freq_YY,dpc_frequency_sum
0,dpc_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,23.0,22.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.043478,0.0,0.0,0.0,0.0,1.0
1,dpc_2,GGGGGGGGGGGGGGG,B,15.0,15.0,14.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
2,dpc_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,17.0,7.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
3,dpc_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,19.0,19.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
4,dpc_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,18.0,10.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0


## Inspect descriptor columns

In [8]:

dpc_count_cols = [c for c in df_dpc.columns if c.startswith("dpc_count_")]
dpc_freq_cols = [c for c in df_dpc.columns if c.startswith("dpc_freq_")]

len(dpc_count_cols), len(dpc_freq_cols)


(400, 400)

In [9]:

df_dpc[["sequence_id", "dpc_length", "dpc_total_dipeptides", "dpc_unique_dipeptides", "dpc_frequency_sum"]].head()


,sequence_id,dpc_length,dpc_total_dipeptides,dpc_unique_dipeptides,dpc_frequency_sum
0,dpc_1,24.0,23.0,22.0,1.0
1,dpc_2,15.0,14.0,1.0,1.0
2,dpc_3,18.0,17.0,7.0,1.0
3,dpc_4,20.0,19.0,19.0,1.0
4,dpc_5,19.0,18.0,10.0,1.0


## Sparse DPC view for one sequence

In [10]:

example_row = df_dpc.loc[0, dpc_freq_cols]
example_nonzero = example_row[example_row > 0].sort_values(ascending=False)
example_nonzero.head(20)


dpc_freq_LF    0.086957
dpc_freq_AY    0.043478
dpc_freq_FL    0.043478
dpc_freq_FR    0.043478
dpc_freq_FS    0.043478
dpc_freq_FI    0.043478
dpc_freq_GV    0.043478
dpc_freq_IS    0.043478
dpc_freq_KW    0.043478
dpc_freq_LL    0.043478
dpc_freq_MK    0.043478
dpc_freq_RG    0.043478
dpc_freq_RR    0.043478
dpc_freq_SA    0.043478
dpc_freq_SL    0.043478
dpc_freq_SR    0.043478
dpc_freq_SS    0.043478
dpc_freq_TF    0.043478
dpc_freq_VF    0.043478
dpc_freq_VT    0.043478
Name: 0, dtype: float64

## Dataset-level DPC summary

In [11]:

dpc_dataset_summary = (
    df_dpc[dpc_freq_cols]
    .mean(axis=0)
    .sort_values(ascending=False)
    .rename("mean_frequency")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

dpc_dataset_summary.head(15)


,descriptor,mean_frequency
0,dpc_freq_GG,0.166667
1,dpc_freq_RR,0.046462
2,dpc_freq_SS,0.044283
3,dpc_freq_KR,0.039216
4,dpc_freq_DD,0.029412
5,dpc_freq_RK,0.029412
6,dpc_freq_PP,0.027778
7,dpc_freq_TT,0.027778
8,dpc_freq_ST,0.026365
9,dpc_freq_DE,0.018576


## Long-format view of the most abundant dipeptides

In [12]:

top_dpc = dpc_dataset_summary.head(20).copy()
top_dpc["dipeptide"] = top_dpc["descriptor"].str.replace("dpc_freq_", "", regex=False)
top_dpc


,descriptor,mean_frequency,dipeptide
0,dpc_freq_GG,0.166667,GG
1,dpc_freq_RR,0.046462,RR
2,dpc_freq_SS,0.044283,SS
3,dpc_freq_KR,0.039216,KR
4,dpc_freq_DD,0.029412,DD
5,dpc_freq_RK,0.029412,RK
6,dpc_freq_PP,0.027778,PP
7,dpc_freq_TT,0.027778,TT
8,dpc_freq_ST,0.026365,ST
9,dpc_freq_DE,0.018576,DE


## Sanity checks

In [13]:

assert len(dpc_count_cols) == 400
assert len(dpc_freq_cols) == 400
assert "dpc_total_dipeptides" in df_dpc.columns
assert "dpc_unique_dipeptides" in df_dpc.columns

valid_rows = df_dpc["dpc_total_dipeptides"] > 0
assert np.allclose(df_dpc.loc[valid_rows, "dpc_frequency_sum"], 1.0)

print(f"Number of DPC count descriptors: {len(dpc_count_cols)}")
print(f"Number of DPC frequency descriptors: {len(dpc_freq_cols)}")
print("DPC implementation sanity checks passed.")


Number of DPC count descriptors: 400
Number of DPC frequency descriptors: 400
DPC implementation sanity checks passed.


## Class-style implementation closer to the real package

In [14]:

class DipeptideCompositionDescriptors:
    """Example class-style DPC implementation for later migration into Roxy."""

    def __init__(self, include_counts: bool = True, include_frequencies: bool = True):
        self.include_counts = include_counts
        self.include_frequencies = include_frequencies

    def transform_sequence(self, seq: str) -> dict:
        return dipeptide_composition(
            seq,
            include_counts=self.include_counts,
            include_frequencies=self.include_frequencies,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


dpc_transformer = DipeptideCompositionDescriptors(include_counts=True, include_frequencies=True)
dpc_matrix = dpc_transformer.transform(df_demo["sequence"].tolist())
dpc_matrix.head()


,dpc_length,dpc_valid_residue_count,dpc_total_dipeptides,dpc_unique_dipeptides,dpc_count_AA,dpc_count_AC,dpc_count_AD,dpc_count_AE,dpc_count_AF,dpc_count_AG,...,dpc_freq_YN,dpc_freq_YP,dpc_freq_YQ,dpc_freq_YR,dpc_freq_YS,dpc_freq_YT,dpc_freq_YV,dpc_freq_YW,dpc_freq_YY,dpc_frequency_sum
0,24,24,23,22,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.043478,0.0,0.0,0.0,0.0,1.0
1,15,15,14,1,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
2,18,18,17,7,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
3,20,20,19,19,0,1,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
4,19,19,18,10,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0


## Merge transformer output back to the dataset

In [15]:

df_dpc_class = pd.concat([df_demo, dpc_matrix], axis=1)
df_dpc_class.head()


,sequence_id,sequence,label,dpc_length,dpc_valid_residue_count,dpc_total_dipeptides,dpc_unique_dipeptides,dpc_count_AA,dpc_count_AC,dpc_count_AD,...,dpc_freq_YN,dpc_freq_YP,dpc_freq_YQ,dpc_freq_YR,dpc_freq_YS,dpc_freq_YT,dpc_freq_YV,dpc_freq_YW,dpc_freq_YY,dpc_frequency_sum
0,dpc_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,23,22,0,0,0,...,0.0,0.0,0.0,0.0,0.043478,0.0,0.0,0.0,0.0,1.0
1,dpc_2,GGGGGGGGGGGGGGG,B,15,15,14,1,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
2,dpc_3,KRRKRRKRRKRRDDDDEE,A,18,18,17,7,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
3,dpc_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,19,19,0,1,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0
4,dpc_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,18,10,0,0,0,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,1.0



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- keep amino-acid constants in `roxy/core/constants.py`
- move DPC helper logic into `roxy/sequence/kmers.py`
- expose a class such as `DipeptideCompositionDescriptors`
- support multiple output modes:
  - counts only
  - frequencies only
  - counts + frequencies
- keep the 400-dimensional ordering fixed and documented
- add tests for:
  - empty sequence
  - sequence of length 1
  - repeated single-residue sequences
  - mixed-composition sequences
  - lower-case input
  - invalid characters removed during cleaning



## Optional export

Uncomment the next cell if you want to save the DPC descriptor table.


In [ ]:
# df_dpc.to_csv("demo_dpc_descriptors.csv", index=False)
